In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np
from scipy.stats import zscore

np.random.seed(42)

# ================================
# 2. CREATE DUMMY DATA (WITH BAD DATA)
# ================================
df = pd.DataFrame({
    "customer_id": [1,2,2,4,5,6,7,8,9,10],  # duplicate ID
    "name": ["Budi", "Ani", "ANI", "Joko", None, "Siti", "BUDI", "Rina", "Rina", "Agus"],
    "age": [25, -5, 30, 200, 40, None, 35, 28, 28, 17],  # invalid age
    "income": [5000, 7000, None, -1000, 12000, 15000, 9999999, 8000, 8000, 4000],  # negative + outlier
    "balance": [10000, 20000, 15000, 30000, None, 50000, 20000000, 25000, 25000, 1000],
    "loan": [2000, 3000, 2500, 1000000, 5000, None, 100, 4000, 4000, 500],
    "gender": ["M", "F", None, "Male", "Female", "F", "m", "female", "female", "X"],  # inconsistent
    "date": ["2024-01-01", "2024/02/01", "invalid", "2024-03-01", None,
             "2024-05-01", "2024-06-01", "2024-07-01", "2024-07-01", "2024-08-01"]
})

print("=== RAW DATA ===")
print(df)

# ================================
# 3. DATA PROFILING
# ================================
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATES ===")
print(df.duplicated().sum())

# ================================
# 4. DATA CLEANSING
# ================================

# ---- 4.1 Remove Duplicates ----
df = df.drop_duplicates()

# ---- 4.2 Standardize Text ----
df["name"] = df["name"].str.lower()

# normalize gender
df["gender"] = df["gender"].str.lower()
df["gender"] = df["gender"].replace({
    "m": "male",
    "f": "female",
    "x": "unknown"
})

# ---- 4.3 Handle Missing Values ----
df["income"].fillna(df["income"].median(), inplace=True)
df["balance"].fillna(df["balance"].median(), inplace=True)
df["loan"].fillna(df["loan"].median(), inplace=True)
df["age"].fillna(df["age"].median(), inplace=True)
df["gender"].fillna("unknown", inplace=True)
df["name"].fillna("unknown", inplace=True)

# ---- 4.4 Fix Date Format ----
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ---- 4.5 Remove Invalid Values ----
df = df[df["age"].between(18, 65)]           # valid age
df = df[df["income"] >= 0]                   # no negative income
df = df[df["balance"] >= 0]

# ---- 4.6 Remove Outliers (Z-Score) ----
numeric_cols = ["income", "balance", "loan"]

z_scores = np.abs(zscore(df[numeric_cols]))
df = df[(z_scores < 3).all(axis=1)]

# ---- 4.7 Business Rules ----
# loan should not exceed 5x income
df = df[df["loan"] <= df["income"] * 5]

# ---- 4.8 Create Data Quality Flags ----
df["flag_missing"] = df.isnull().sum(axis=1)
df["flag_clean"] = (df["flag_missing"] == 0).astype(int)

# ================================
# 5. FINAL OUTPUT
# ================================
print("\n=== CLEAN DATA ===")
print(df)

print("\n=== FINAL DATA QUALITY ===")
print(df.isnull().sum())

print("\n=== SUMMARY ===")
print(f"Final rows: {len(df)}")